# Baseline Remaining-Time Regression — PoC

Single-model proof-of-concept: `RandomForestRegressor` with randomised hyperparameter search.
Uses only log-based features (`BasicControlFlowFeatures`). Time-series features are excluded.

In [ ]:
import contextlib
import logging
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline as SklearnPipeline
from tqdm import tqdm

from spi_time_series.pipeline.checkpointing import checkpoint
from spi_time_series.data import Dataset
from spi_time_series.data.constants import VALID_END_ACTIVITIES
from spi_time_series.data.schemas import (
    EventLog,
    ModelArtifact,
    PreprocessedData,
    RawData,
)
from spi_time_series.evaluation.metrics import evaluate
from spi_time_series.features.extraction import extract_features_builder
from spi_time_series.features.log_based_features import BasicControlFlowFeatures
from spi_time_series.preprocessing.preprocess import (
    _build_trace_samples,
    clean_data,
    sliding_window_factory,
    split_data,
)

logging.basicConfig(level=logging.INFO, force=True)

## Config

All tunable values live in the two cells below.

In [2]:
RANDOM_STATE = 42
CV_FOLDS = 3
N_ITER = 10
MIN_PREFIX_LENGTH = 3
# Cap each trace at its MAX_PREFIX_LENGTH most recent events. Without a cap, BPI 2017
# generates ~800K training prefixes (avg trace length ~38, min_length=3). A cap of 10
# reduces this to ~220K and cuts extraction time proportionally. Remove or raise the cap
# once the pipeline is validated end-to-end.
MAX_PREFIX_LENGTH = 10

CHECKPOINT_DIR = Path("../data/checkpoints")

In [3]:
HP_GRID = {
    "n_estimators": [50, 100, 150, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "max_features": ["sqrt", 0.5],
}

In [4]:
# Cumulative params dicts — each stage inherits all upstream params so that a change
# to any upstream setting also invalidates downstream checkpoints.
CLEANED_PARAMS = {
    "valid_end_activities": sorted(VALID_END_ACTIVITIES),
}

FEATURE_PARAMS = {
    **CLEANED_PARAMS,
    "min_prefix_length": MIN_PREFIX_LENGTH,
    "max_prefix_length": MAX_PREFIX_LENGTH,
    "feature_classes": [BasicControlFlowFeatures.__name__],
}

ARTIFACT_PARAMS = {
    **FEATURE_PARAMS,
    "model": RandomForestRegressor.__name__,
    "hp_grid": HP_GRID,
    "n_iter": N_ITER,
    "cv_folds": CV_FOLDS,
    "random_state": RANDOM_STATE,
}

## tqdm_joblib helper

Patches joblib's batch-completion callback so that every completed parallel job advances a tqdm bar.
Works with both the CV search and the final model fit (both use joblib internally).
No additional packages required — `tqdm` and `joblib` are already project dependencies.

In [5]:
@contextlib.contextmanager
def tqdm_joblib(tqdm_object):
    """Context manager: show a tqdm bar for joblib parallel work."""

    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback
    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old

## Target generator

The `TargetGenerator` protocol `(trace, start_idx, end_idx) -> float` has no `col_idx` parameter.
A module-level dict `_col_idx_store` is populated by the splitter (which runs before feature extraction)
and read by `remaining_time_hours` — safe because pipeline stages run sequentially.

In [ ]:
_col_idx_store: dict[str, int] = {}


def remaining_time_hours(trace, start_idx: int, end_idx: int) -> float:
    """Hours from the last event in the prefix to the final event of the trace."""
    ts_idx = _col_idx_store["time:timestamp"]
    delta = trace[-1, ts_idx] - trace[end_idx - 1, ts_idx]
    return delta.total_seconds() / 3600


def my_preprocessor(raw: RawData) -> EventLog:
    cleaned = clean_data(raw, valid_ends=VALID_END_ACTIVITIES)
    return cleaned.event_log


def my_splitter(log: EventLog) -> PreprocessedData:
    train_df, test_df = split_data(log)

    _col_idx_store.clear()
    _col_idx_store.update({c: i for i, c in enumerate(train_df.columns)})

    prefix_gen = sliding_window_factory(
        min_length=MIN_PREFIX_LENGTH,
        max_length=MAX_PREFIX_LENGTH,
    )

    return PreprocessedData(
        train_log=_build_trace_samples(train_df, prefix_gen),
        num_train_cases=len(train_df["case:concept:name"].unique()),
        test_log=_build_trace_samples(test_df, prefix_gen),
        num_test_cases=len(test_df["case:concept:name"].unique()),
        col_idx=_col_idx_store.copy(),
    )

## Stage 1 — Load & preprocess

In [7]:
dataset = Dataset()
raw = RawData(event_log=dataset.log)

cleaned = checkpoint(
    CHECKPOINT_DIR / "cleaned_log.pkl",
    lambda: my_preprocessor(raw),
    params=CLEANED_PARAMS,
)
preprocessed = my_splitter(cleaned)

INFO:spi_time_series.data.dataset:Dataset found at G:\spi-time-series\data\raw\BPI Challenge 2017.xes.gz.
INFO:spi_time_series.data.dataset:Reading XES log...
g:\spi-time-series\.venv\Lib\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(
INFO:spi_time_series.data.dataset:Done. Loaded 1202267 events.
INFO:spi_time_series.checkpointing:Loading checkpoint: ..\data\checkpoints\cleaned_log__9ea2b74a.pkl
INFO:spi_time_series.preprocessing.preprocess:Splitting cases at cutoff time: 2016-10-18 17:03:13.107600128+00:00
INFO:spi_time_series.preprocessing.preprocess:Split successful. 
Train: 22723 cases, 
Test: 6247 cases.


## Stage 2 — Feature extraction

`one_hot_encode_categorical` is left **off** (default). Calling `sklearn.OneHotEncoder.transform`
on a single row per prefix has high per-call overhead; with ~800K prefixes on BPI 2017 this alone
adds ~1.6M sklearn calls. The bag-of-activities columns (`count_<activity>`) already capture
which activities appeared and how many times, so the information loss is acceptable for a PoC.
OHE can be re-enabled later once the pipeline is validated end-to-end.

In [8]:
feature_extractor = extract_features_builder(
    [BasicControlFlowFeatures()],
    remaining_time_hours,
)

features = checkpoint(
    CHECKPOINT_DIR / "features.pkl",
    lambda: feature_extractor(preprocessed),
    params=FEATURE_PARAMS,
)

print(f"Train shape: {features.X_train.shape}")
print(f"Test shape:  {features.X_test.shape}")

INFO:spi_time_series.checkpointing:Loading checkpoint: ..\data\checkpoints\features__c9afd2c5.pkl


Train shape: (822706, 33)
Test shape:  (227073, 33)


## Stages 3 & 4 — Hyperparameter search + fit

Both stages are wrapped in `_fit_artifact()` so they can be skipped as a unit when the artifact checkpoint exists.
`refit=False` keeps the CV search and final model fit as separate, clearly labelled progress bars.

In [ ]:
def _fit_artifact() -> ModelArtifact:
    preprocessor = ColumnTransformer(
        [
            (
                "num",
                SimpleImputer(strategy="median"),
                make_column_selector(dtype_include=np.number),
            )
        ],
        remainder="drop",
    )
    X_train_num = preprocessor.fit_transform(features.X_train)

    search = RandomizedSearchCV(
        RandomForestRegressor(n_jobs=-1, random_state=RANDOM_STATE),
        param_distributions=HP_GRID,
        n_iter=N_ITER,
        cv=CV_FOLDS,
        scoring="neg_mean_absolute_error",
        refit=False,
        random_state=RANDOM_STATE,
    )

    # total=None: tqdm_joblib counts joblib batch completions, not CV fold evaluations.
    # With n_jobs=-1, RandomForest spawns many small tree-building batches per fold so
    # the count would far exceed N_ITER * CV_FOLDS. An indeterminate bar is more honest.
    with tqdm_joblib(tqdm(desc="Hyperparameter search", total=None)):
        search.fit(X_train_num, features.y_train)

    print(f"\nBest params: {search.best_params_}")
    print(f"Best CV MAE: {-search.best_score_:.2f} h")

    best_rf = RandomForestRegressor(
        **search.best_params_, n_jobs=-1, random_state=RANDOM_STATE
    )
    with tqdm_joblib(tqdm(desc="Fitting best model", total=None)):
        best_rf.fit(X_train_num, features.y_train)

    fitted_pipeline = SklearnPipeline(
        [
            ("preprocessor", preprocessor),
            ("model", best_rf),
        ]
    )
    return ModelArtifact(
        models={"random_forest": fitted_pipeline},
        feature_names=features.feature_names,
        target_col="remaining_time_hours",
    )

In [ ]:
artifact = checkpoint(
    CHECKPOINT_DIR / "artifact.pkl", _fit_artifact, params=ARTIFACT_PARAMS
)

INFO:spi_time_series.checkpointing:Loading checkpoint: ..\data\checkpoints\artifact__67fedc1b.pkl


## Stage 5 — Evaluate

In [11]:
report = evaluate(artifact, features)

INFO:spi_time_series.evaluation.metrics:Evaluating model: random_forest
INFO:spi_time_series.evaluation.metrics:Model random_forest evaluated across 8 prefix lengths.


## Results

MAE, RMSE (hours), and R² per prefix length.

In [12]:
rows = [
    {
        "prefix_length": pl,
        "MAE (h)": round(report.metrics["random_forest"][pl]["mae"], 3),
        "RMSE (h)": round(report.metrics["random_forest"][pl]["rmse"], 3),
        "R²": round(report.metrics["random_forest"][pl]["r2"], 4),
    }
    for pl in report.prefix_lengths
]

pd.DataFrame(rows)

,prefix_length,MAE (h),RMSE (h),R²
0,3,248.868,305.491,-0.1343
1,4,240.332,284.964,0.0122
2,5,240.640,285.724,0.0069
3,6,240.535,286.203,0.0050
4,7,238.239,283.212,0.0131
5,8,237.717,282.776,0.0161
6,9,237.549,281.713,0.0133
7,10,156.974,220.922,0.3791
